# Notebook 1: Tu primer grafo con LangGraph

En este notebook aprenderás los conceptos más fundamentales de LangGraph:

- Qué es un `StateGraph` y cómo funciona el **estado compartido**
- Cómo definir **nodos** (funciones que transforman el estado)
- Cómo conectar nodos con **edges** (aristas)
- Cómo **compilar** y **ejecutar** el grafo

Construiremos un agente muy simple: recibe una pregunta → la procesa con Gemini → devuelve la respuesta.

## 1. Instalación de dependencias

In [ ]:
# %pip install -qU langgraph langchain-google-genai

## 2. Configuración de la API Key

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

## 3. Conceptos clave antes de empezar

### ¿Qué es un grafo en LangGraph?

LangGraph modela flujos de trabajo como **grafos dirigidos**:

```
[START] → [nodo_1] → [nodo_2] → [END]
```

- **Nodos**: funciones Python que reciben el estado actual y retornan una actualización del estado.
- **Edges**: conexiones entre nodos que determinan el flujo de ejecución.
- **Estado**: un diccionario (o clase TypedDict) que se pasa entre nodos y acumula la información.

### El estado es la clave

El estado es el "hilo conductor" del grafo. Cada nodo lo lee y lo modifica. Necesitamos definirlo con `TypedDict`.

## 4. Definir el Estado

In [ ]:
from typing import TypedDict

# El estado define qué información viaja a través del grafo.
# Cada campo puede ser leído y escrito por los nodos.
class EstadoAgente(TypedDict):
    pregunta: str       # La pregunta del usuario
    respuesta: str      # La respuesta generada por el modelo
    procesado: bool     # Si la pregunta ya fue procesada

print("Estado definido correctamente:", EstadoAgente.__annotations__)

## 5. Inicializar el modelo LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Inicializamos Gemini 2.5 Flash-Lite
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.7,
)

print("Modelo inicializado:", llm.model)

## 6. Definir los Nodos

Cada nodo es una función que:
1. Recibe el estado actual (como diccionario)
2. Realiza alguna operación
3. Retorna un diccionario con las **claves del estado que quiere actualizar**

In [ ]:
# Nodo 1: Preprocesar la pregunta
def nodo_preprocesar(estado: EstadoAgente) -> dict:
    """
    Este nodo limpia y prepara la pregunta antes de enviarla al LLM.
    Retorna solo los campos que quiere modificar.
    """
    pregunta_limpia = estado["pregunta"].strip().rstrip("?!") + "?"
    print(f"[Nodo Preprocesar] Pregunta original: '{estado['pregunta']}'")
    print(f"[Nodo Preprocesar] Pregunta limpia:   '{pregunta_limpia}'")
    
    return {
        "pregunta": pregunta_limpia,
        "procesado": False  # Aún no hemos procesado con el LLM
    }


# Nodo 2: Llamar al LLM
def nodo_responder(estado: EstadoAgente) -> dict:
    """
    Este nodo envía la pregunta al LLM y guarda la respuesta en el estado.
    """
    print(f"[Nodo Responder] Enviando al LLM: '{estado['pregunta']}'")
    
    respuesta = llm.invoke(estado["pregunta"])
    
    print(f"[Nodo Responder] Respuesta recibida (primeros 80 chars): '{respuesta.content[:80]}...'")
    
    return {
        "respuesta": respuesta.content,
        "procesado": True
    }


print("Nodos definidos: nodo_preprocesar, nodo_responder")

## 7. Construir el Grafo

Ahora ensamblamos todo: estado + nodos + edges.

In [ ]:
from langgraph.graph import StateGraph, START, END

# 1. Crear el grafo con nuestro estado
grafo = StateGraph(EstadoAgente)

# 2. Agregar los nodos
grafo.add_node("preprocesar", nodo_preprocesar)
grafo.add_node("responder",   nodo_responder)

# 3. Definir las conexiones (edges)
#    START → preprocesar → responder → END
grafo.add_edge(START, "preprocesar")
grafo.add_edge("preprocesar", "responder")
grafo.add_edge("responder", END)

# 4. Compilar el grafo (lo convierte en un objeto ejecutable)
app = grafo.compile()

print("✅ Grafo compilado correctamente")

## 8. Visualizar el grafo (opcional)

Si tienes `pygraphviz` instalado, puedes ver el grafo gráficamente.

In [ ]:
# Visualización en ASCII (siempre disponible)
print(app.get_graph().draw_ascii())

In [ ]:
# Visualización con Mermaid (si estás en Jupyter/Colab)
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Visualización gráfica no disponible:", e)
    print("Usa draw_ascii() en su lugar")

## 9. Ejecutar el grafo

In [ ]:
# Estado inicial: solo necesitamos la pregunta.
# Los demás campos se irán llenando a medida que el grafo avanza.
estado_inicial = {
    "pregunta": "¿Qué es la fotosíntesis",
    "respuesta": "",
    "procesado": False
}

print("=" * 60)
print("Ejecutando el grafo...")
print("=" * 60)

resultado = app.invoke(estado_inicial)

print("=" * 60)
print("Estado final del grafo:")
print("=" * 60)
print(f"Pregunta:  {resultado['pregunta']}")
print(f"Procesado: {resultado['procesado']}")
print(f"Respuesta:\n{resultado['respuesta']}")

## 10. Streaming: ver el estado nodo por nodo

Una de las ventajas de LangGraph es poder observar cómo evoluciona el estado en cada paso.

In [ ]:
estado_inicial_2 = {
    "pregunta": "¿Cuántos planetas hay en el sistema solar?",
    "respuesta": "",
    "procesado": False
}

print("Ejecutando con streaming de estados...")
print("=" * 60)

for paso in app.stream(estado_inicial_2):
    # 'paso' es un diccionario: {nombre_nodo: estado_actualizado}
    for nombre_nodo, estado_actualizado in paso.items():
        print(f"\n📌 Nodo ejecutado: '{nombre_nodo}'")
        print(f"   Estado actualizado: {estado_actualizado}")

## 11. Ejercicios propuestos

Ahora que entiendes lo básico, intenta estos ejercicios:

1. **Agrega un tercer nodo** llamado `nodo_formatear` que tome la respuesta del LLM y la ponga en mayúsculas. Conéctalo entre `responder` y `END`.

2. **Añade un campo `idioma`** al estado. En `nodo_preprocesar`, agrega al prompt la instrucción "Responde en [idioma]".

3. **Mide el tiempo**: añade un campo `tiempo_ms` al estado y en `nodo_responder` calcula cuánto tardó la llamada al LLM.

## Resumen

| Concepto | Descripción |
|---|---|
| `TypedDict` | Define la estructura del estado compartido |
| `StateGraph` | El contenedor del grafo |
| `add_node` | Registra una función como nodo |
| `add_edge` | Conecta dos nodos en secuencia |
| `compile()` | Convierte el grafo en un ejecutable |
| `invoke()` | Ejecuta el grafo con un estado inicial |
| `stream()` | Ejecuta el grafo paso a paso, retornando cada estado intermedio |